# Week 3: Encapsulation — Protecting Object State
### PHASE 1: Building One Reliable Component

*📚 Object Oriented Programming · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Explain what **encapsulation** means and why it matters
2. Identify the problem of **invalid object state**
3. Use the **underscore convention** (`_`) to mark private attributes
4. Create **properties** with `@property` for controlled read access
5. Write **setters with validation** to prevent bad data
6. Design **read-only properties** that cannot be changed from outside
7. Build a class that **protects its own state** from invalid values

## 🎯 Core Mastery Connection

A reliable component protects its own state. Invalid data should be impossible — the component enforces its own rules. When objects collaborate in a composed system, each one must guarantee its own correctness. If a Motor accepts speed=500%, the entire system built on top of it becomes unreliable. Encapsulation is what makes a component *trustworthy*.

---
## Part 1: What is Encapsulation?

**Encapsulation** means keeping an object's data safe by controlling how it is accessed and changed.

Think of a **motor controller** on a robot:
- You can set the speed (0 to 100%)
- You should NOT be able to set the speed to 500% (the motor would burn!)
- The controller **protects** the motor by checking values before applying them

That is encapsulation: **the object controls its own data**.

| Without Encapsulation | With Encapsulation |
|---|---|
| Anyone can change any value | The object checks values before accepting |
| Invalid states are possible | Invalid states are prevented |
| Bugs are hard to find | Errors are caught immediately |
| Like an open circuit board | Like a sealed controller box |

---
## Part 2: The Problem — Invalid State

Let's see what happens when we **don't** protect our data.

**Figure 3.1** — A motor class without any protection:

In [ ]:
# Figure 3.1 - Motor without encapsulation

class Motor:
    def __init__(self, name):
        self.name = name
        self.speed = 0          # speed in percent (should be 0-100)
        self.temperature = 25   # temperature in Celsius

    def status(self):
        print(f"{self.name}: speed={self.speed}%, temp={self.temperature}C")

# Create a motor
m = Motor("Left Wheel")
m.status()

Now let's break it by setting impossible values:

**Figure 3.2** — Creating invalid state:

In [ ]:
# Figure 3.2 - Setting invalid values (bad!)

m.speed = 500           # Impossible! Max should be 100
m.temperature = -300    # Impossible! Below absolute zero
m.status()              # The object is now in an INVALID state

The motor accepted speed=500% and temperature=-300C. In a real robot, this could cause damage!

**The problem:** Anyone can set any value, and the object cannot protect itself.

---
## Part 3: Private Attributes (Convention: `_underscore`)

In Python, we use a **naming convention** to say "this attribute is private":

| Convention | Meaning | Example |
|---|---|---|
| `self.name` | Public — anyone can use it | `self.name = "Motor A"` |
| `self._speed` | Private — should not be used from outside | `self._speed = 50` |

The single underscore `_` means: **"Please don't touch this directly. Use the proper methods instead."**

It is a **convention** (a polite request), not a hard rule. Python trusts the programmer.

**Figure 3.3** — Using underscore convention:

In [ ]:
# Figure 3.3 - Private attributes with underscore

class Motor:
    def __init__(self, name):
        self.name = name       # public: OK to access directly
        self._speed = 0        # private: should not be accessed directly
        self._temperature = 25 # private: should not be accessed directly

    def status(self):
        print(f"{self.name}: speed={self._speed}%, temp={self._temperature}C")

m = Motor("Left Wheel")
m.status()

# This still WORKS (Python doesn't enforce it)
# but it is BAD PRACTICE:
# m._speed = 500  # Don't do this!

But now we need a **proper way** to read and change these private values. That's where **properties** come in.

---
## Part 4: Properties — Controlled Access (`@property`)

A **property** lets you read a private attribute in a controlled way.

It looks like a regular attribute from the outside, but it runs a method inside.

**Figure 3.4** — Adding a property to read speed:

In [ ]:
# Figure 3.4 - Using @property for controlled reading

class Motor:
    def __init__(self, name):
        self.name = name
        self._speed = 0

    @property
    def speed(self):
        """Read the motor speed."""
        return self._speed

# Create motor and read speed
m = Motor("Left Wheel")

# Reading works - it calls the property method
print(f"Speed is: {m.speed}%")

# Writing does NOT work yet - we get an error!
# m.speed = 50  # AttributeError: can't set attribute

Notice:
- We write `m.speed` (no parentheses) — it **looks like** an attribute
- But it **runs a method** behind the scenes
- Without a setter, the property is **read-only** by default

---
## Part 5: Setter with Validation (`@name.setter`)

To allow **writing** with validation, we add a **setter**.

The setter is a method that:
1. Receives the new value
2. **Checks** if the value is valid
3. Only saves it if it passes the check

**Figure 3.5** — Adding a setter with validation:

In [ ]:
# Figure 3.5 - Property with getter AND setter

class Motor:
    def __init__(self, name):
        self.name = name
        self._speed = 0

    @property
    def speed(self):
        """Read the motor speed (0-100)."""
        return self._speed

    @speed.setter
    def speed(self, value):
        """Set the motor speed with validation."""
        if value < 0:
            print(f"Error: Speed cannot be negative ({value})")
            return
        if value > 100:
            print(f"Error: Speed cannot exceed 100% ({value})")
            return
        self._speed = value
        print(f"Speed set to {value}%")

# Test it
m = Motor("Left Wheel")

m.speed = 50    # Valid
m.speed = 75    # Valid
m.speed = 500   # Invalid - rejected!
m.speed = -10   # Invalid - rejected!

print(f"Final speed: {m.speed}%")  # Still 75

Now the motor **protects itself**! It will not accept invalid speed values.

**The pattern:**
```python
@property
def name(self):          # Getter - for reading
    return self._name

@name.setter
def name(self, value):   # Setter - for writing with validation
    # check value here
    self._name = value
```

**Figure 3.6** — Another example with a temperature sensor:

In [ ]:
# Figure 3.6 - Temperature sensor with validation

class TemperatureSensor:
    def __init__(self, location):
        self.location = location
        self._reading = 0.0    # in Celsius
        self._min_temp = -40   # sensor hardware limit
        self._max_temp = 125   # sensor hardware limit

    @property
    def reading(self):
        """Get the current temperature reading."""
        return self._reading

    @reading.setter
    def reading(self, value):
        """Set reading only if within sensor range."""
        if value < self._min_temp or value > self._max_temp:
            print(f"Error: {value}C is outside sensor range "
                  f"({self._min_temp} to {self._max_temp})")
            return
        self._reading = value

# Test
sensor = TemperatureSensor("Engine Bay")
sensor.reading = 85.5     # Valid
print(f"Temperature: {sensor.reading}C")

sensor.reading = 200      # Invalid - rejected
print(f"Temperature: {sensor.reading}C")  # Still 85.5

---
## Part 6: Read-Only Properties

Sometimes you want a value that can be **read but never changed** from outside.

Just create a `@property` **without** a setter.

**Figure 3.7** — Read-only properties:

In [ ]:
# Figure 3.7 - Read-only properties

class Battery:
    def __init__(self, capacity_mah):
        self._capacity = capacity_mah   # Cannot change after creation
        self._charge = capacity_mah     # Starts fully charged

    @property
    def capacity(self):
        """Battery capacity - read only (cannot change)."""
        return self._capacity

    @property
    def charge(self):
        """Current charge level."""
        return self._charge

    @property
    def percentage(self):
        """Charge as percentage - calculated, read only."""
        return round(self._charge / self._capacity * 100, 1)

    def use(self, amount):
        """Use some battery charge."""
        self._charge = max(0, self._charge - amount)

# Test
bat = Battery(5000)
print(f"Capacity: {bat.capacity} mAh")
print(f"Charge: {bat.percentage}%")

bat.use(2000)
print(f"Charge after use: {bat.percentage}%")

# This would cause an error:
# bat.capacity = 99999  # AttributeError: can't set attribute

Notice `percentage` is a **calculated** read-only property. It doesn't store a value — it computes one from other data.

| Property Type | Has `@property`? | Has `@x.setter`? | Can Read? | Can Write? |
|---|---|---|---|---|
| Read-only | Yes | No | Yes | No |
| Read-write with validation | Yes | Yes | Yes | Yes (with checks) |
| Calculated | Yes | No | Yes (computed) | No |

---
## Part 7: Putting It All Together

Let's build a complete class that uses everything we learned.

**Figure 3.8** — A servo motor controller with full encapsulation:

In [ ]:
# Figure 3.8 - Complete encapsulated class: ServoMotor

class ServoMotor:
    """A servo motor that can rotate between 0 and 180 degrees."""

    def __init__(self, name, min_angle=0, max_angle=180):
        self.name = name
        self._min_angle = min_angle    # read-only
        self._max_angle = max_angle    # read-only
        self._angle = 90               # start at center
        self._is_enabled = False       # motor starts off

    # --- Read-only properties ---

    @property
    def min_angle(self):
        return self._min_angle

    @property
    def max_angle(self):
        return self._max_angle

    @property
    def is_enabled(self):
        return self._is_enabled

    # --- Read-write property with validation ---

    @property
    def angle(self):
        return self._angle

    @angle.setter
    def angle(self, value):
        if not self._is_enabled:
            print(f"Error: {self.name} is not enabled. Call enable() first.")
            return
        if value < self._min_angle or value > self._max_angle:
            print(f"Error: Angle {value} is out of range "
                  f"({self._min_angle}-{self._max_angle})")
            return
        self._angle = value
        print(f"{self.name} moved to {value} degrees")

    # --- Methods ---

    def enable(self):
        self._is_enabled = True
        print(f"{self.name} enabled")

    def disable(self):
        self._is_enabled = False
        print(f"{self.name} disabled")

    def center(self):
        """Move to the center position."""
        if self._is_enabled:
            mid = (self._min_angle + self._max_angle) // 2
            self.angle = mid

# Test the servo
arm = ServoMotor("Robot Arm", 0, 180)

arm.angle = 45         # Error: not enabled
arm.enable()
arm.angle = 45         # Works
arm.angle = 200        # Error: out of range
arm.center()           # Move to 90

print(f"\nAngle: {arm.angle}")
print(f"Range: {arm.min_angle} to {arm.max_angle}")
print(f"Enabled: {arm.is_enabled}")

---
## 🎢 Exercises — Components That Protect Themselves

> **Composition preview:** In a composed system, each component must enforce its own rules. A `MonitoringSystem` should not have to check whether its `Sensor` accepted a bad value — the Sensor should reject it on its own. Every class you build below is a self-defending component ready for composition.

Complete the exercises below. Each exercise has a difficulty level:
- **Easy** — Direct application of what you learned
- **Medium** — Requires some thinking
- **Challenge** — Combines multiple concepts

---
## 🎢 Exercises

Complete the exercises below. Each exercise has a difficulty level:
- **Easy** — Direct application of what you learned
- **Medium** — Requires some thinking
- **Challenge** — Combines multiple concepts

### Exercise 1 (Easy)
Create a `LED` class with a `_brightness` attribute (0 to 255).
Add a `@property` getter and a `@brightness.setter` that rejects values outside 0-255.

<details>
<summary>💡 Hint</summary>
Use <code>@property</code> for the getter and <code>@brightness.setter</code> with a check like <code>if 0 <= value <= 100</code>.
</details>

In [ ]:
# ✏️ [EX1] LED with brightness validation



### Exercise 2 (Easy)
Create a `PressureSensor` class with a read-only property `reading`.
The reading should only be changeable through an `update(value)` method that checks if the value is between 0 and 1013 (millibars).

<details>
<summary>💡 Hint</summary>
Store the raw value in <code>self._pressure</code>. In the setter, validate it's within the sensor's physical range.
</details>

In [ ]:
# ✏️ [EX2] PressureSensor with read-only property



### Exercise 3 (Easy)
Create a `WaterTank` class with:
- `_capacity` (liters, read-only, set in `__init__`)
- `_level` (current water level)
- A `level` property with a setter that ensures level is between 0 and capacity

<details>
<summary>💡 Hint</summary>
Track <code>self._level</code> and <code>self._capacity</code>. The <code>fill()</code> method should not exceed capacity.
</details>

In [ ]:
# ✏️ [EX3] WaterTank with capacity limit



### Exercise 4 (Easy)
Create a `Counter` class with:
- A read-only `count` property
- Methods `increment()` and `reset()`
- The count should never go below 0

<details>
<summary>💡 Hint</summary>
Use <code>self._count</code> with a property. <code>increment()</code> adds 1, <code>reset()</code> sets to 0. Prevent negative values.
</details>

In [ ]:
# ✏️ [EX4] Counter with read-only count



### Exercise 5 (Medium)
Create a `DCMotor` class with:
- `speed` property (0 to 100)
- `direction` property (only `"forward"` or `"reverse"`)
- Both must validate their values

<details>
<summary>💡 Hint</summary>
Store <code>self._speed</code>. The setter should clamp values between 0 and <code>self._max_speed</code>.
</details>

In [ ]:
# ✏️ [EX5] DCMotor with speed and direction validation



### Exercise 6 (Medium)
Create a `Thermostat` class with:
- `target_temp` property (settable, must be between 15 and 30)
- `current_temp` (read-only, updated by `measure(value)` method)
- A calculated read-only property `is_heating` that returns `True` if current < target

<details>
<summary>💡 Hint</summary>
Use a property for <code>target_temp</code>. The setter validates the range (e.g., 15-30°C). Add a <code>status</code> read-only property.
</details>

In [ ]:
# ✏️ [EX6] Thermostat with calculated property



### Exercise 7 (Medium)
Create a `RobotGripper` class with:
- `opening` property (0 to 100 mm)
- `force` property (0 to 50 Newtons)
- A read-only `is_gripping` property that is `True` when opening < 10 and force > 5

<details>
<summary>💡 Hint</summary>
Use <code>self._position</code> (0-100). The setter clamps to valid range. <code>grip()</code> sets to 100, <code>release()</code> sets to 0.
</details>

In [ ]:
# ✏️ [EX7] RobotGripper with multiple validated properties



### Exercise 8 (Medium)
Create a `MotorDriver` class where:
- `speed` can only be changed if the motor `is_enabled`
- There are `enable()` and `disable()` methods
- When disabled, speed is automatically set to 0
- Speed range is 0-100

<details>
<summary>💡 Hint</summary>
Encapsulate <code>self._enabled</code> and <code>self._duty_cycle</code>. Only allow duty_cycle changes when enabled.
</details>

In [ ]:
# ✏️ [EX8] MotorDriver with enable/disable logic



### Exercise 9 (Challenge)
Create a `PasswordProtectedMotor` class with:
- A `_password` set during creation
- A `speed` property (0 to 100)
- Speed can only be changed if you call `unlock(password)` first
- After setting speed, it automatically locks again
- A read-only `is_locked` property

<details>
<summary>💡 Hint</summary>
Store a hashed password (use simple comparison for now). Require password in setter methods. Raise <code>ValueError</code> on wrong password.
</details>

In [ ]:
# ✏️ [EX9] PasswordProtectedMotor



### Exercise 10 (Challenge)
Create a `StepperMotor` class with:
- `_position` (starts at 0, read-only property)
- `_step_size` (degrees per step, read-only, set in `__init__`)
- `step_forward()` and `step_backward()` methods
- Position must stay within -360 to 360 degrees
- A calculated property `revolution_count` that returns how many full revolutions have been made (position / 360)

<details>
<summary>💡 Hint</summary>
Track <code>self._position</code> (integer steps). <code>step_forward(n)</code> and <code>step_backward(n)</code> with bounds checking.
</details>

In [ ]:
# ✏️ [EX10] StepperMotor with position tracking



### Exercise 11 (Challenge)
Create a `SmartHeater` class with:
- `_power` (0-100%, validated property)
- `_max_power` (set in __init__, read-only)
- `_temperature` (read-only, starts at 20)
- A `tick()` method that simulates one time step: temperature increases by `power * 0.1` but decreases by 2 (cooling)
- Temperature should never go below 0
- A read-only `is_overheating` property that returns `True` if temperature > 100

<details>
<summary>💡 Hint</summary>
Combine a temperature sensor property with a heater state. Add logic: if temp < target, turn on; if temp > target + 2, turn off.
</details>

In [ ]:
# ✏️ [EX11] SmartHeater with simulation



---
### 🌉 Bridge to Next Week

This week we learned to **protect** the data inside a single object using encapsulation.

But real systems are made of **many objects working together**:
- A robot has motors, sensors, and controllers
- A sensor system has a sensor, a filter, and a logger

Next week, we will learn **Composition** — how to build complex systems by combining simple objects. An object can **contain** other objects, and each one does its own job.

See you in **Week 4: Composition — Building with Objects**!

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_03"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")